In [1]:
import yfinance as yf
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Ask the user to input the ticker first
while True:
    ticker_symbol = input("Enter ticker symbol (e.g., 0700.HK): ").strip().upper()
    if ticker_symbol:
        break
    print("Ticker symbol cannot be empty. Please try again.")

# 1. Get daily data
ticker = yf.Ticker(ticker_symbol)
hist_daily = ticker.history(period="10y", interval="1d", auto_adjust=False)

if hist_daily.empty:
    raise SystemExit(f"No data found for ticker '{ticker_symbol}'. Please check the symbol and try again.")

# Convert index to datetime and remove timezone if present
hist_daily.index = pd.to_datetime(hist_daily.index)
if hist_daily.index.tz is not None:
    hist_daily.index = hist_daily.index.tz_localize(None)

# 2. Filter out future dates if any
today = pd.Timestamp.today().normalize()
hist_daily = hist_daily[hist_daily.index <= today]

# 3. Function to get the actual date and Close price
def get_actual_period_data(df, freq):
    grouped = df.groupby(pd.Grouper(freq=freq))

    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )

    result = result.dropna(subset=['Actual_Date'])
    result = result.reset_index(drop=True)
    return result

# 4. Get quarter-end data
quarterly_data = get_actual_period_data(hist_daily, 'QE')

# =========================================================
# TRANSPOSE PROCESS (LEFT TO RIGHT BASED ON MOST RECENT DATE)
# =========================================================

# 1. Sort by Actual_Date from newest to oldest
quarterly_data = quarterly_data.sort_values(by='Actual_Date', ascending=False)

# 2. Set Actual_Date as index, then transpose
# so dates become columns from left to right
quarterly_data_transposed = quarterly_data.set_index('Actual_Date').T

# 3. Display result
print(f"\n--- {ticker_symbol} Quarter-End Closing Prices (Left to Right: Most Recent) ---")
quarterly_data_transposed

Enter ticker symbol (e.g., 0700.HK): 0700.HK

--- 0700.HK Quarter-End Closing Prices (Left to Right: Most Recent) ---


Actual_Date,2026-09-22,2026-06-30,2026-03-31,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31,2024-09-30,2024-06-28,2024-03-28,2023-12-29,2023-09-29,2023-06-30,2023-03-31,2022-12-30,2022-09-30,2022-06-30,2022-03-31,2021-12-31,2021-09-30,2021-06-30,2021-03-31,2020-12-31,2020-09-30,2020-06-30,2020-03-31,2019-12-31,2019-09-30,2019-06-28,2019-03-29,2018-12-31,2018-09-28,2018-06-29,2018-03-29,2017-12-29,2017-09-29,2017-06-30,2017-03-31,2016-12-30,2016-09-30
Close,451.600006,429.799988,484.0,599.0,663.0,503.0,497.0,419.799988,444.600006,372.399994,303.799988,293.600006,306.200012,331.600006,385.799988,317.226044,253.021011,336.601532,355.407135,421.104034,425.344574,538.364136,562.332458,521.770752,471.529572,459.637634,350.489807,346.249268,304.397003,325.04657,332.790192,289.462921,297.944,363.027069,377.592407,374.273712,309.928131,257.382324,205.389618,174.876175,196.355423


In [2]:
import yfinance as yf
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Ask the user to input the ticker first
while True:
    ticker_symbol = input("Enter ticker symbol (e.g., 0700.HK): ").strip().upper()
    if ticker_symbol:
        break
    print("Ticker symbol cannot be empty. Please try again.")

# 1. Get daily data
ticker = yf.Ticker(ticker_symbol)
hist_daily = ticker.history(period="10y", interval="1d", auto_adjust=False)

if hist_daily.empty:
    raise SystemExit(f"No data found for ticker '{ticker_symbol}'. Please check the symbol and try again.")

# Convert index to datetime and remove timezone if present
hist_daily.index = pd.to_datetime(hist_daily.index)
if hist_daily.index.tz is not None:
    hist_daily.index = hist_daily.index.tz_localize(None)

# 2. Filter out future dates if any
today = pd.Timestamp.today().normalize()
hist_daily = hist_daily[hist_daily.index <= today]

# 3. Function to get the actual date and Close price
def get_actual_period_data(df, freq):
    grouped = df.groupby(pd.Grouper(freq=freq))

    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )

    result = result.dropna(subset=['Actual_Date'])
    result = result.reset_index(drop=True)
    return result

# 4. Get quarter-end data
quarterly_data = get_actual_period_data(hist_daily, 'QE')

# 5. Function to create period label
def get_period_label(date_str):
    dt = pd.to_datetime(date_str)
    y = dt.year
    m = dt.month

    if m == 3:
        return f"1Q{y}"
    elif m == 6:
        return f"1H{y}"
    elif m == 9:
        return f"9M{y}"
    elif m == 12:
        return f"FY{y}"
    else:
        # Fallback if not a standard quarter-end
        return f"{m}M{y}"

# 6. Add period label column
quarterly_data['Period'] = quarterly_data['Actual_Date'].apply(get_period_label)

# =========================================================
# TRANSPOSE PROCESS (LEFT TO RIGHT BASED ON MOST RECENT DATE)
# =========================================================

# Sort by Actual_Date from newest to oldest
quarterly_data = quarterly_data.sort_values(by='Actual_Date', ascending=False)

# Set Period as index, then transpose so Period becomes column headers
quarterly_data_transposed = quarterly_data.set_index('Period').T

# Ensure row order: Actual_Date then Close
quarterly_data_transposed = quarterly_data_transposed.reindex(['Actual_Date', 'Close'])

# =========================================================
# REMOVE COLUMN NAME (THE "PERIOD" TEXT IN THE TOP LEFT CORNER)
# =========================================================
quarterly_data_transposed.index.name = None
quarterly_data_transposed.columns.name = None  # This removes the "Period" header text

# Display result
print(f"\n--- {ticker_symbol} Quarter-End Closing Prices (Left to Right: Most Recent) ---")

# If using Jupyter Notebook, simply use the following line as the final output:
quarterly_data_transposed

Enter ticker symbol (e.g., 0700.HK): 0700.HK

--- 0700.HK Quarter-End Closing Prices (Left to Right: Most Recent) ---


,9M2026,1H2026,1Q2026,FY2025,9M2025,1H2025,1Q2025,FY2024,9M2024,1H2024,1Q2024,FY2023,9M2023,1H2023,1Q2023,FY2022,9M2022,1H2022,1Q2022,FY2021,9M2021,1H2021,1Q2021,FY2020,9M2020,1H2020,1Q2020,FY2019,9M2019,1H2019,1Q2019,FY2018,9M2018,1H2018,1Q2018,FY2017,9M2017,1H2017,1Q2017,FY2016,9M2016
Actual_Date,2026-09-22,2026-06-30,2026-03-31,2025-12-31,2025-09-30,2025-06-30,2025-03-31,2024-12-31,2024-09-30,2024-06-28,2024-03-28,2023-12-29,2023-09-29,2023-06-30,2023-03-31,2022-12-30,2022-09-30,2022-06-30,2022-03-31,2021-12-31,2021-09-30,2021-06-30,2021-03-31,2020-12-31,2020-09-30,2020-06-30,2020-03-31,2019-12-31,2019-09-30,2019-06-28,2019-03-29,2018-12-31,2018-09-28,2018-06-29,2018-03-29,2017-12-29,2017-09-29,2017-06-30,2017-03-31,2016-12-30,2016-09-30
Close,451.600006,429.799988,484.0,599.0,663.0,503.0,497.0,419.799988,444.600006,372.399994,303.799988,293.600006,306.200012,331.600006,385.799988,317.226044,253.021011,336.601532,355.407135,421.104034,425.344574,538.364136,562.332458,521.770752,471.529572,459.637634,350.489807,346.249268,304.397003,325.04657,332.790192,289.462921,297.944,363.027069,377.592407,374.273712,309.928131,257.382324,205.389618,174.876175,196.355423


In [3]:
import yfinance as yf
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Ask the user to input the ticker first
while True:
    ticker_symbol = input("Enter ticker symbol (e.g., 0700.HK): ").strip().upper()
    if ticker_symbol:
        break
    print("Ticker symbol cannot be empty. Please try again.")

# Ask the user to choose reporting frequency
while True:
    freq_choice = input("Choose reporting frequency - [Q]uarterly or [S]emiannual: ").strip().upper()
    if freq_choice in ('Q', 'QUARTER', 'QUARTERLY'):
        freq_choice = 'Q'
        break
    elif freq_choice in ('S', 'SEMI', 'SEMIANNUAL', 'H', 'HALF', 'HALFYEARLY'):
        freq_choice = 'S'
        break
    print("Invalid choice. Please enter Q for quarterly or S for semiannual.")

# 1. Get daily data
ticker = yf.Ticker(ticker_symbol)
hist_daily = ticker.history(period="10y", interval="1d", auto_adjust=False)

if hist_daily.empty:
    raise SystemExit(f"No data found for ticker '{ticker_symbol}'. Please check the symbol and try again.")

# Convert index to datetime and remove timezone if present
hist_daily.index = pd.to_datetime(hist_daily.index)
if hist_daily.index.tz is not None:
    hist_daily.index = hist_daily.index.tz_localize(None)

# 2. Filter out future dates if any
today = pd.Timestamp.today().normalize()
hist_daily = hist_daily[hist_daily.index <= today]

# 3. Function to get actual date and Close price for quarterly data
def get_actual_period_data(df, freq):
    grouped = df.groupby(pd.Grouper(freq=freq))
    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )
    result = result.dropna(subset=['Actual_Date'])
    result = result.reset_index(drop=True)
    return result

# 4. Function to get semiannual data (June and December only)
def get_semiannual_data(df):
    # Keep only June and December (typical half-year ends)
    mask = df.index.month.isin([6, 12])
    filtered = df[mask]
    if filtered.empty:
        return pd.DataFrame(columns=['Actual_Date', 'Close'])

    # Group by year and month to get the last trading day in each June/December
    grouped = filtered.groupby([filtered.index.year, filtered.index.month])
    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )
    result = result.dropna(subset=['Actual_Date']).reset_index(drop=True)
    return result

# 5. Get period data based on user choice
if freq_choice == 'Q':
    period_data = get_actual_period_data(hist_daily, 'QE')
else:
    period_data = get_semiannual_data(hist_daily)

if period_data.empty:
    freq_label = 'quarterly' if freq_choice == 'Q' else 'semiannual'
    raise SystemExit(f"No {freq_label} data found for ticker '{ticker_symbol}'.")

# 6. Function to create period label
def get_period_label(date_str):
    dt = pd.to_datetime(date_str)
    y = dt.year
    m = dt.month

    if m == 3:
        return f"1Q{y}"
    elif m == 6:
        return f"1H{y}"
    elif m == 9:
        return f"9M{y}"
    elif m == 12:
        return f"FY{y}"
    else:
        # Fallback if not a standard quarter-end
        return f"{m}M{y}"

# 7. Add period label column
period_data['Period'] = period_data['Actual_Date'].apply(get_period_label)

# =========================================================
# TRANSPOSE PROCESS (LEFT TO RIGHT BASED ON MOST RECENT DATE)
# =========================================================

# Sort by Actual_Date from newest to oldest
period_data = period_data.sort_values(by='Actual_Date', ascending=False)

# Set Period as index, then transpose so Period becomes column headers
period_data_transposed = period_data.set_index('Period').T

# Ensure row order: Actual_Date then Close
period_data_transposed = period_data_transposed.reindex(['Actual_Date', 'Close'])

# =========================================================
# REMOVE COLUMN NAME (THE "PERIOD" TEXT IN THE TOP LEFT CORNER)
# =========================================================
period_data_transposed.index.name = None
period_data_transposed.columns.name = None  # Removes the "Period" header text

# Display result
freq_label = 'Quarter-End' if freq_choice == 'Q' else 'Semiannual'
print(f"\n--- {ticker_symbol} {freq_label} Closing Prices (Left to Right: Most Recent) ---")

# If using Jupyter Notebook, you can use the following line as the final output instead of print:
period_data_transposed

Enter ticker symbol (e.g., 0700.HK): 0700.HK
Choose reporting frequency - [Q]uarterly or [S]emiannual: S

--- 0700.HK Semiannual Closing Prices (Left to Right: Most Recent) ---


,1H2026,FY2025,1H2025,FY2024,1H2024,FY2023,1H2023,FY2022,1H2022,FY2021,1H2021,FY2020,1H2020,FY2019,1H2019,FY2018,1H2018,FY2017,1H2017,FY2016
Actual_Date,2026-06-30,2025-12-31,2025-06-30,2024-12-31,2024-06-28,2023-12-29,2023-06-30,2022-12-30,2022-06-30,2021-12-31,2021-06-30,2020-12-31,2020-06-30,2019-12-31,2019-06-28,2018-12-31,2018-06-29,2017-12-29,2017-06-30,2016-12-30
Close,429.799988,599.0,503.0,419.799988,372.399994,293.600006,331.600006,317.226044,336.601532,421.104034,538.364136,521.770752,459.637634,346.249268,325.04657,289.462921,363.027069,374.273712,257.382324,174.876175


In [4]:
import yfinance as yf
import pandas as pd

# Show all columns without truncation
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Ask the user to input the ticker first
while True:
    ticker_symbol = input("Enter ticker symbol (e.g., 0700.HK): ").strip().upper()
    if ticker_symbol:
        break
    print("Ticker symbol cannot be empty. Please try again.")

# Ask the user to choose reporting frequency
while True:
    freq_choice = input("Choose reporting frequency - [Q]uarterly or [S]emiannual: ").strip().upper()
    if freq_choice in ('Q', 'QUARTER', 'QUARTERLY'):
        freq_choice = 'Q'
        break
    elif freq_choice in ('S', 'SEMI', 'SEMIANNUAL', 'H', 'HALF', 'HALFYEARLY'):
        freq_choice = 'S'
        break
    print("Invalid choice. Please enter Q for quarterly or S for semiannual.")

# 1. Get daily data
ticker = yf.Ticker(ticker_symbol)
hist_daily = ticker.history(period="10y", interval="1d", auto_adjust=False)

if hist_daily.empty:
    raise SystemExit(f"No data found for ticker '{ticker_symbol}'. Please check the symbol and try again.")

# Convert index to datetime and remove timezone if present
hist_daily.index = pd.to_datetime(hist_daily.index)
if hist_daily.index.tz is not None:
    hist_daily.index = hist_daily.index.tz_localize(None)

# 2. Filter out future dates if any
today = pd.Timestamp.today().normalize()
hist_daily = hist_daily[hist_daily.index <= today]

# 3. Extract latest closing price (most recent trading day)
latest_date = hist_daily.index[-1]
latest_date_str = latest_date.strftime('%Y-%m-%d')
latest_close = hist_daily['Close'].iloc[-1]

# Create a DataFrame for the latest close (will become the first column)
latest_df = pd.DataFrame(
    {latest_date_str: [latest_date_str, latest_close]},
    index=['Actual_Date', 'Close']
)

# 4. Function to get actual date and Close price for quarterly data
def get_actual_period_data(df, freq):
    grouped = df.groupby(pd.Grouper(freq=freq))
    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )
    result = result.dropna(subset=['Actual_Date'])
    result = result.reset_index(drop=True)
    return result

# 5. Function to get semiannual data (June and December only)
def get_semiannual_data(df):
    # Keep only June and December (typical half-year ends)
    mask = df.index.month.isin([6, 12])
    filtered = df[mask]
    if filtered.empty:
        return pd.DataFrame(columns=['Actual_Date', 'Close'])

    # Group by year and month to get the last trading day in each June/December
    grouped = filtered.groupby([filtered.index.year, filtered.index.month])
    result = grouped.agg(
        Actual_Date=('Close', lambda x: x.index[-1].strftime('%Y-%m-%d') if not x.empty else None),
        Close=('Close', 'last')
    )
    result = result.dropna(subset=['Actual_Date']).reset_index(drop=True)
    return result

# 6. Get period data based on user choice
if freq_choice == 'Q':
    period_data = get_actual_period_data(hist_daily, 'QE')
else:
    period_data = get_semiannual_data(hist_daily)

if period_data.empty:
    freq_label = 'quarterly' if freq_choice == 'Q' else 'semiannual'
    raise SystemExit(f"No {freq_label} data found for ticker '{ticker_symbol}'.")

# 7. Function to create period label
def get_period_label(date_str):
    dt = pd.to_datetime(date_str)
    y = dt.year
    m = dt.month

    if m == 3:
        return f"1Q{y}"
    elif m == 6:
        return f"1H{y}"
    elif m == 9:
        return f"9M{y}"
    elif m == 12:
        return f"FY{y}"
    else:
        # Fallback if not a standard quarter-end
        return f"{m}M{y}"

# 8. Add period label column
period_data['Period'] = period_data['Actual_Date'].apply(get_period_label)

# =========================================================
# TRANSPOSE PROCESS (LEFT TO RIGHT BASED ON MOST RECENT DATE)
# =========================================================

# Sort by Actual_Date from newest to oldest
period_data = period_data.sort_values(by='Actual_Date', ascending=False)

# Set Period as index, then transpose so Period becomes column headers
period_data_transposed = period_data.set_index('Period').T

# Ensure row order: Actual_Date then Close
period_data_transposed = period_data_transposed.reindex(['Actual_Date', 'Close'])

# =========================================================
# ADD LATEST CLOSE AS THE FIRST COLUMN
# =========================================================
final_df = pd.concat([latest_df, period_data_transposed], axis=1)

# =========================================================
# REMOVE COLUMN NAME (THE "PERIOD" TEXT IN THE TOP LEFT CORNER)
# =========================================================
final_df.index.name = None
final_df.columns.name = None

# Display result
freq_label = 'Quarter-End' if freq_choice == 'Q' else 'Semiannual'
print(f"\n--- {ticker_symbol} {freq_label} Closing Prices + Latest Close ({latest_date_str}) ---")
print("(Left to Right: Latest Close, then Most Recent Period)")

# If using Jupyter Notebook, you can use the following line as the final output instead of print:
final_df

Enter ticker symbol (e.g., 0700.HK): 0700.HK
Choose reporting frequency - [Q]uarterly or [S]emiannual: S

--- 0700.HK Semiannual Closing Prices + Latest Close (2026-09-22) ---
(Left to Right: Latest Close, then Most Recent Period)


,2026-09-22,1H2026,FY2025,1H2025,FY2024,1H2024,FY2023,1H2023,FY2022,1H2022,FY2021,1H2021,FY2020,1H2020,FY2019,1H2019,FY2018,1H2018,FY2017,1H2017,FY2016
Actual_Date,2026-09-22,2026-06-30,2025-12-31,2025-06-30,2024-12-31,2024-06-28,2023-12-29,2023-06-30,2022-12-30,2022-06-30,2021-12-31,2021-06-30,2020-12-31,2020-06-30,2019-12-31,2019-06-28,2018-12-31,2018-06-29,2017-12-29,2017-06-30,2016-12-30
Close,451.600006,429.799988,599.0,503.0,419.799988,372.399994,293.600006,331.600006,317.226044,336.601532,421.104034,538.364136,521.770752,459.637634,346.249268,325.04657,289.462921,363.027069,374.273712,257.382324,174.876175
